# Day 51 — Explainable AI for Customer Churn

This notebook explains why the capstone's churn model produces predictions using global feature importance and SHAP where supported.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))
from src.preprocessing import engineer_features, build_preprocessor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
df = engineer_features(pd.read_csv(ROOT/'data/raw/customer_data.csv'))
X=df.drop(columns=['churn','customer_id']); y=df['churn']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
preprocessor=build_preprocessor(X_train)
model=Pipeline([('preprocessor',preprocessor),('classifier',RandomForestClassifier(n_estimators=200,max_depth=10,class_weight='balanced',random_state=42))])
model.fit(X_train,y_train)
print('Model fitted')

## 1. Global feature importance

Random Forest impurity-based importance gives a model-level view of which transformed variables contribute most to splits across the ensemble.

In [ ]:
rf=model.named_steps['classifier']
prep=model.named_steps['preprocessor']
names=prep.get_feature_names_out()
importance=pd.Series(rf.feature_importances_,index=names).sort_values(ascending=False)
importance.head(15)

## 2. SHAP explanations

SHAP attributes prediction differences to input features. For tree models, TreeExplainer can provide global and local explanations. If the local environment has a SHAP compatibility issue, the accompanying script records the issue and still produces model feature importance.

In [ ]:
try:
    import shap
    X_trans=prep.transform(X_test)
    explainer=shap.TreeExplainer(rf)
    vals=explainer.shap_values(X_trans)
    vals = vals[1] if isinstance(vals,list) else (vals[:,:,1] if getattr(vals,'ndim',0)==3 else vals)
    shap_global=pd.DataFrame({'feature':names,'mean_abs_shap':np.abs(vals).mean(axis=0)}).sort_values('mean_abs_shap',ascending=False)
    shap_global.head(15)
except Exception as e:
    print('SHAP unavailable in this environment:', type(e).__name__, e)

## 3. Business interpretation

Feature importance tells us which variables the model relies on; it does **not** by itself prove causation. Business teams should interpret high-importance variables as predictive signals and validate interventions with experiments and domain knowledge.